# 허깅페이스에서 모델받아 다국어번역 서비스만들기

In [4]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install sentencepiece

   ---------------------------------------- 0.0/991.5 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/991.5 kB ? eta -:--:--
   ---------------------------------------- 991.5/991.5 kB 3.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

ko_text = "키움 히어로즈 파이팅!"
chinese_text = "生活就像一盒巧克力。"

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# translate korean to english
tokenizer.src_lang = "ko"
encoded_hi = tokenizer(ko_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_hi, forced_bos_token_id=tokenizer.get_lang_id("en"))
result1 =tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result1)
# => "La vie est comme une boîte de chocolat."

# translate 한국어 to 일본어
tokenizer.src_lang = "ko"
encoded_zh = tokenizer(ko_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_zh, forced_bos_token_id=tokenizer.get_lang_id("ja"))
result2=tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result2)
# => "Life is like a box of chocolate."


['The Heroes Fighting!']
['ヒーローズの戦い!']


In [3]:
import gradio as gr
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# 모델 및 토크나이저 로딩
model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# 지원 언어 목록 (ISO 639-1 코드 기준)
lang_dict = {
    "영어 (English)": "en",
    "일본어 (Japanese)": "ja",
    "중국어 (Chinese)": "zh",
    "프랑스어 (French)": "fr",
    "스페인어 (Spanish)": "es",
    "독일어 (German)": "de",
    "베트남어 (Vietnamese)": "vi"
    # 필요시 더 추가 가능
}

def translate(text, target_lang_label):
    if not text.strip():
        return ""
    
    target_lang = lang_dict[target_lang_label]
    tokenizer.src_lang = "ko"  # 입력은 한국어 기준

    encoded = tokenizer(text, return_tensors="pt")
    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(target_lang)
    )
    result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return result[0]

# Gradio 인터페이스
iface = gr.Interface(
    fn=translate,
    inputs=[
        gr.Textbox(label="원문 (한국어)", lines=4, placeholder="번역할 한국어 문장을 입력하세요."),
        gr.Dropdown(choices=list(lang_dict.keys()), label="번역 언어 선택")
    ],
    outputs=gr.Textbox(label="번역 결과", lines=4),
    title="다국어 번역기 (M2M100)",
    description="한국어를 다양한 언어로 번역합니다. Hugging Face M2M100 모델 사용."
)

iface.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [4]:
iface.close()

Closing server running on port: 7860


In [8]:
pip install --upgrade transformers sentencepiece datasets[audio]


   ---------------------------------------- 0.0/25.8 MB ? eta -:--:--
   -- ------------------------------------- 1.6/25.8 MB 8.4 MB/s eta 0:00:03
   ------ --------------------------------- 4.5/25.8 MB 11.2 MB/s eta 0:00:02
   ---------- ----------------------------- 6.6/25.8 MB 10.6 MB/s eta 0:00:02
   --------------- ------------------------ 10.2/25.8 MB 12.3 MB/s eta 0:00:02
   --------------------- ------------------ 13.9/25.8 MB 13.9 MB/s eta 0:00:01
   ------------------------- -------------- 16.5/25.8 MB 13.2 MB/s eta 0:00:01
   ---------------------------- ----------- 18.6/25.8 MB 13.1 MB/s eta 0:00:01
   ----------------------------- ---------- 18.9/25.8 MB 12.7 MB/s eta 0:00:01
   ----------------------------- ---------- 18.9/25.8 MB 12.7 MB/s eta 0:00:01
   ----------------------------- ---------- 18.9/25.8 MB 12.7 MB/s eta 0:00:01
   ----------------------------- ---------- 18.9/25.8 MB 12.7 MB/s eta 0:00:01
   ----------------------------- ---------- 18.9/25.8 MB 12.7 MB/

  You can safely remove it manually.
  You can safely remove it manually.


In [6]:
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts")

embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
# You can replace this embedding with your own as well.

speech = synthesiser("Hello, my dog is cooler than you!", forward_params={"speaker_embeddings": speaker_embedding})

sf.write("speech.wav", speech["audio"], samplerate=speech["sampling_rate"])


ModuleNotFoundError: No module named 'datasets'

In [5]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset
import torch
import soundfile as sf
from datasets import load_dataset

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

inputs = processor(text="Hello, my dog is cute.", return_tensors="pt")

# load xvector containing speaker's voice characteristics from a dataset
embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embeddings = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)

speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)

sf.write("speech.wav", speech.numpy(), samplerate=16000)


ModuleNotFoundError: No module named 'datasets'